# LFW and DroneFace: SFace extraction inside OP-TEE

This notebook loops over the images from `ppface`, using the existing real trusted application for every SFace inference. Raw embeddings stay inside the TA. Linux prepares the face crop; the TA extracts 128 features, applies `int(feature * 256)` in the 64-bit ring, masks, and returns only masked coordinates and timings.

**Current boundary:** image loading, OpenCV face detection/eye alignment and resize occur in Linux. This is a dataset experiment, not protected camera capture. The TA's existing fresh pads are discarded. Exported files are benchmark artifacts, **not yet Obscura dealer-compatible inputs**. Masking, TA commands and protocol files are unchanged.

See [DATASET_WORKFLOW.md](DATASET_WORKFLOW.md) for the source notebook analysis, installation and proposed protocol integration. The optional `verify_native.py` is an outside-TEE validation tool, not used by this notebook.


In [ ]:
from pathlib import Path
import sys
import json
from datetime import datetime, timezone

# Works when Jupyter starts in the repository root or jetson_benchmark.
BENCHMARK = Path.cwd() if (Path.cwd() / "dataset_tee.py").is_file() else Path.cwd() / "jetson_benchmark"
if not (BENCHMARK / "dataset_tee.py").is_file():
    raise FileNotFoundError("Set BENCHMARK to the folder containing dataset_tee.py")
sys.path.insert(0, str(BENCHMARK.resolve()))
from dataset_tee import collect_images, run_dataset


## Configuration

Point `PPFACE` at the actual dataset directory on the Jetson. It must contain `lfw` and `DnHFaces/open_data_set`. The original notebook's last SFace experiment uses `trio_cam` and all probes. Set `EASY_PROBES=True` to reproduce the image selection of its preceding cell (heights 3/4, distance at most 30).

Start with 40 images per dataset. With the default gallery and ordering this includes 33 DroneFace portraits and seven probes. After the smoke run, set `LIMIT_PER_DATASET=None` for all images and rerun from this configuration cell to obtain a fresh output directory. Each image is inferred once, after two warmups per dataset. No embedding cache bypasses the TEE. Full LFW can take hours; `OFFSET` and `LIMIT_PER_DATASET` allow deterministic chunks. To repeat the image selection of an Obscura pair list, set its entry in `PAIR_FILES`. A reused image gets one measured extraction. For DroneFace the companion gallery portraits are included for each selected gallery subject.


In [ ]:
# Look beside the repository, including when the benchmark is in dist/jetson_test.
PPFACE = next((p / "ppface" for p in BENCHMARK.resolve().parents
               if (p / "ppface/lfw").is_dir() and (p / "ppface/DnHFaces").is_dir()),
              BENCHMARK.parent / "ppface")  # edit if your datasets are elsewhere
PAIR_FILES = {"lfw": None, "droneface": None}  # None selects the full dataset
# PAIR_FILES["lfw"] = PPFACE.parent / "data/veri_test1.txt"
# PAIR_FILES["droneface"] = PPFACE.parent / "data/veri_balanced_cosine9sface.txt"
GALLERY = "trio_cam"
EASY_PROBES = False
DATASETS = ("droneface", "lfw")
LIMIT_PER_DATASET = 40  # change to None for the complete dataset loops
OFFSET = 0
POWER = True  # False explicitly disables energy reporting, not TEE execution
RUN_ROOT = BENCHMARK / "results" / ("dataset_tee_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ"))

for dataset in DATASETS:
    _, inventory = collect_images(PPFACE, dataset, GALLERY, EASY_PROBES, PAIR_FILES[dataset])
    print(json.dumps(inventory, indent=2))
print("Output directory:", RUN_ROOT)


## Extract each dataset inside the TEE

`run_dataset` implements the image loop: hash the source → detect/align/resize in Linux → invoke the TA on the 112×112 BGR crop → persist masked coordinates and timing metadata. It reuses a single TA session per dataset, but performs actual neural inference for every image. If OP-TEE fails, the run stops; there is no Linux inference fallback.

The original notebook wrote raw float embeddings into JSON `.txt` files. Here `.txt` files contain an explicitly different, versioned format with **masked uint64 values as decimal strings**. Do not copy them over `data/new_emb1.txt` or pass them to `convert_raw`.


In [ ]:
summaries = {}
for dataset in DATASETS:
    summaries[dataset] = run_dataset(
        PPFACE, dataset, RUN_ROOT / dataset,
        gallery=GALLERY, easy_probes=EASY_PROBES, pairs=PAIR_FILES[dataset],
        limit=LIMIT_PER_DATASET, offset=OFFSET, warmup=2, power=POWER,
    )
    print(json.dumps(summaries[dataset], indent=2))


## Read measurements, without displaying biometric vectors

`ta_inference_ms` measures neural kernels inside OP-TEE. `pipeline_ms` also includes weight transfers, hashing, transitions, scaling, masking and clearing. The coarse TA counters may read zero for short phases. `prepare_ms` is Linux image hashing/preparation. Energy is module-rail energy over the entire dataset loop, including preparation and record writing; it is not isolated TEE or per-process energy.

The files `records.jsonl` preserve completed records if a run fails. `manifest.json` records complete/failed status. Only a successful run has `summary.json`. Use a new output directory for each run. No masks are reused.


In [ ]:
for dataset in DATASETS:
    folder = RUN_ROOT / dataset
    result = json.loads((folder / "summary.json").read_text())
    print(dataset, "images:", result["images"])
    print("Mean secure inference ms:", result["timings_ms"]["ta_inference_ms"]["mean"])
    print("Mean complete TEE call ms:", result["timings_ms"]["pipeline_ms"]["mean"])
    print("Module energy:", result["power"])
    print("Masked output:", folder / f"masked_embeddings_{dataset}.txt")


## Before the Obscura integration

This notebook does not modify the dealer or expose raw embeddings. The next reviewed change must provision matching dealer masks privately into the TA, use each setup slot once, and connect masked outputs to the same multiplication/FSS material, including the secure self-product. The existing independent pads do not meet that requirement.

DroneFace portrait averaging must also be resolved before verification: the old notebook averages three embeddings but stores a single portrait key. Averaging already masked exports is not a replacement for averaging the raw vectors inside the trusted boundary before fixed-point conversion. See the proposal in `DATASET_WORKFLOW.md` before changing any masking or import/export protocol.
